
# Repeated simulation benchmark: when MS-AA disentangles latent network motifs better than PCA

This notebook extends the first successful simulation into a **proper controlled benchmark** using the same `Subject`, `multi_subject_aa(...)`, spatial orientation, and `sXC @ S` reconstruction logic as the empirical MS-AA notebooks.

The goal is not to show that MS-AA universally outperforms PCA. Instead, the benchmark tests a more specific theoretical claim:

> **When observations are generated as convex mixtures of overlapping, nonorthogonal extremal network motifs, PCA can preserve the relevant low-dimensional subspace while its individual variance directions rotate/mix the generating motifs. MS-AA should more directly recover the individual extremal motifs and their network/subnetwork composition.**

The notebook tests five things:

1. **Motif overlap / nonorthogonality** — the primary theoretical manipulation.
2. **Measurement noise** — robustness.
3. **Between-subject heterogeneity** — robustness.
4. **Fitted model order \(K\)** — sensitivity to under/over-specification.
5. **Orthogonal Gaussian latent factors** — a negative-control regime in which PCA should be favored.

For every archetypal simulation, we separately quantify:

- observed-data reconstruction error,
- one-to-one recovery of true motifs,
- subnetwork-level motif recovery,
- PCA-subspace recovery of the true motifs,
- recovered-dimension selectivity/purity,
- off-target mixing.

The notebook is **checkpointed and restart-safe**. Completed simulations are written to CSV after every fit.

---

## Recommended workflow

Start with:

```python
RUN_MODE = "quick"
```

to verify the full pipeline. Then switch to:

```python
RUN_MODE = "paper"
```

for the larger repeated benchmark.

The `"paper"` settings are intentionally substantial because every replicate fits the actual multisubject AA model.


In [ ]:

# ============================================================
# Configuration
# ============================================================

RUN_MODE = "quick"   # "quick" or "paper"
RNG_SEED = 42

HELPERS_DIR = "helpers"
OUTPUT_DIR = "simulation_benchmark_outputs"

ANALYSIS_TYPE = "spatial"

# Same style of options as the empirical fitting notebook.
MSAA_OPTS_BASE = dict(
    maxiter=200,
    conv_crit=1e-6,
    fix_var_iter=5,
    heteroscedastic=False,
    use_gpu=False,
    rngSEED=RNG_SEED,
)

# Ground-truth architecture
N_NETWORKS = 6
SUBNETWORKS_PER_NETWORK = 2
NODES_PER_SUBNETWORK = 10
N_TRUE_MOTIFS = 6

# Base generative settings
BASE_NOISE_SD = 0.25
BASE_MULTIPLICATIVE_SD = 0.08
BASE_ADDITIVE_SD = 0.04
DIRICHLET_ALPHA = 0.10

# Primary manipulation
OVERLAP_LEVELS = [0.0, 0.25, 0.50, 0.75, 1.0]

# Robustness manipulations
NOISE_LEVELS = [0.10, 0.25, 0.40, 0.55]
HETEROGENEITY_LEVELS = [0.0, 0.5, 1.0, 1.5]

# K sensitivity
K_LEVELS = [4, 5, 6, 8, 10]

# Number of repeated fits
if RUN_MODE == "quick":
    N_SUBJECTS = 10
    T_PER_SUBJECT = 100
    N_REPS_MAIN = 2
    N_REPS_ROBUST = 1
    N_REPS_K = 1
    N_REPS_GAUSSIAN = 2

elif RUN_MODE == "paper":
    N_SUBJECTS = 16
    T_PER_SUBJECT = 150
    N_REPS_MAIN = 20
    N_REPS_ROBUST = 10
    N_REPS_K = 10
    N_REPS_GAUSSIAN = 20

else:
    raise ValueError("RUN_MODE must be 'quick' or 'paper'")

print("RUN_MODE:", RUN_MODE)
print("Subjects:", N_SUBJECTS)
print("Time points / subject:", T_PER_SUBJECT)


In [ ]:

# ============================================================
# Imports
# ============================================================

%matplotlib inline

import os
import sys
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.optimize import linear_sum_assignment
from sklearn.decomposition import PCA

sys.path.append(HELPERS_DIR)

# Same MS-AA implementation as the empirical notebooks
from MultiSubject_AA import Subject, multi_subject_aa

OUTPUT_DIR = Path(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_CSV = OUTPUT_DIR / f"benchmark_results_{RUN_MODE}.csv"

print("Results checkpoint:", RESULTS_CSV)



## 1. Exact MS-AA fitting and reconstruction path

For this simulation, matrices are **explicitly generated as \(T \times V\)**. We therefore preserve that orientation directly rather than trying to infer orientation from matrix dimensions.


In [ ]:

def coerce_TxV(x):
    x = np.asarray(x, dtype=float)

    if x.ndim != 2:
        raise ValueError(
            f"Expected a 2D T x V matrix, got {x.shape}"
        )

    # Preserve known simulation orientation.
    x = x - x.mean(axis=0, keepdims=True)

    return x.astype(np.float32, copy=False)


def orient_subject_matrix(X_raw, analysis_type):
    Xtv = coerce_TxV(X_raw)

    return (
        Xtv
        if analysis_type == "spatial"
        else Xtv.T
    )


def build_subject_objects(subject_list, analysis_type):
    return [
        Subject(
            X=orient_subject_matrix(x, analysis_type),
            sX=orient_subject_matrix(x, analysis_type),
        )
        for x in subject_list
    ]


def fit_msaa(subject_list, K, analysis_type="spatial", seed=RNG_SEED, verbose=False):
    opts = dict(MSAA_OPTS_BASE)
    opts["rngSEED"] = int(seed)

    subj_objs = build_subject_objects(
        subject_list,
        analysis_type,
    )

    t0 = time.time()

    results_subj, C, cost_fun, varexpl, secs = multi_subject_aa(
        subj_objs,
        noc=int(K),
        opts=opts,
    )

    elapsed = time.time() - t0

    if verbose:
        print(
            f"[{analysis_type.upper()} AA] "
            f"K={K} | VarExpl={varexpl*100:.2f}% | "
            f"{elapsed:.1f}s"
        )
        print(
            "subj0 sXC:",
            np.asarray(results_subj[0]["sXC"]).shape,
        )
        print(
            "subj0 S:",
            np.asarray(results_subj[0]["S"]).shape,
        )

    return {
        "results_subj": results_subj,
        "C": C,
        "cost_fun": cost_fun,
        "varexpl": varexpl,
        "secs": secs,
        "K": int(K),
        "elapsed": elapsed,
    }


def reconstruct_from_archetypes(
    sub,
    analysis_type="spatial",
    archetype_indices=None,
    zscore=False,
):
    sXC = np.asarray(sub["sXC"], dtype=float)
    S = np.asarray(sub["S"], dtype=float)

    if archetype_indices is None:
        idx = np.arange(S.shape[0])
    else:
        idx = np.asarray(list(archetype_indices), dtype=int)

    if analysis_type == "spatial":
        Xhat = sXC[:, idx] @ S[idx, :]
    elif analysis_type == "temporal":
        Xhat = (sXC[:, idx] @ S[idx, :]).T
    else:
        raise ValueError(
            "analysis_type must be 'spatial' or 'temporal'"
        )

    if zscore:
        Xhat = (
            Xhat - Xhat.mean(axis=0, keepdims=True)
        ) / (
            Xhat.std(axis=0, keepdims=True) + 1e-8
        )

    return Xhat



## 2. Ground-truth motif geometry with controlled overlap

We create two endpoints:

- **Separated endpoint:** each motif is dominated by its own subnetwork.
- **Overlapping endpoint:** motifs are distributed across multiple subnetworks and share some subnetworks with other motifs.

For overlap parameter \(o\),

\[
M(o)=(1-o)M_{\mathrm{separated}} + o M_{\mathrm{overlapping}}.
\]

This lets us manipulate nonorthogonality continuously while preserving the same number of motifs and nodes.

The notebook reports the mean absolute pairwise correlation among the true motifs at every overlap level, so the manipulation is quantified rather than merely labeled.


In [ ]:

def make_motif_endpoints():
    n_subnetworks = (
        N_NETWORKS * SUBNETWORKS_PER_NETWORK
    )
    n_nodes = (
        n_subnetworks * NODES_PER_SUBNETWORK
    )

    separated = np.full(
        (N_TRUE_MOTIFS, n_nodes),
        0.05,
        dtype=float,
    )

    # Unique subnetwork for each motif: 0,2,4,6,8,10
    unique_subnets = [2 * k for k in range(N_TRUE_MOTIFS)]

    for k, subnet in enumerate(unique_subnets):
        lo = subnet * NODES_PER_SUBNETWORK
        hi = lo + NODES_PER_SUBNETWORK
        separated[k, lo:hi] = 1.0

    overlapping = np.full_like(separated, 0.05)

    # Same distributed/overlapping construction as the successful first simulation.
    motif_subnetworks = [
        [0,  3,  8],
        [1,  3,  9],
        [2,  5, 10],
        [2,  4, 11],
        [0,  6,  7],
        [1,  6,  8],
    ]

    for k, active in enumerate(motif_subnetworks):
        for subnet in active:
            lo = subnet * NODES_PER_SUBNETWORK
            hi = lo + NODES_PER_SUBNETWORK
            overlapping[k, lo:hi] = 1.0

    return separated, overlapping


MOTIFS_SEPARATED, MOTIFS_OVERLAPPING = make_motif_endpoints()


def make_true_motifs(overlap):
    overlap = float(overlap)

    motifs = (
        (1.0 - overlap) * MOTIFS_SEPARATED
        + overlap * MOTIFS_OVERLAPPING
    )

    return motifs


def mean_pairwise_abs_corr(patterns):
    C = np.corrcoef(patterns)

    tri = np.triu_indices_from(
        C,
        k=1,
    )

    return np.mean(
        np.abs(C[tri])
    )


geometry_rows = []

for overlap in OVERLAP_LEVELS:
    motifs = make_true_motifs(overlap)

    geometry_rows.append({
        "overlap": overlap,
        "mean_pairwise_abs_corr": mean_pairwise_abs_corr(motifs),
    })

geometry_df = pd.DataFrame(geometry_rows)

display(geometry_df)


In [ ]:

def plot_pattern_matrix(patterns, title, ylabel="Pattern"):
    fig, ax = plt.subplots(figsize=(12, 4))

    im = ax.imshow(
        patterns,
        aspect="auto",
    )

    n_nodes = patterns.shape[1]

    for x in range(
        NODES_PER_SUBNETWORK,
        n_nodes,
        NODES_PER_SUBNETWORK,
    ):
        ax.axvline(
            x - 0.5,
            linewidth=0.4,
            alpha=0.35,
        )

    network_width = (
        SUBNETWORKS_PER_NETWORK
        * NODES_PER_SUBNETWORK
    )

    for x in range(
        network_width,
        n_nodes,
        network_width,
    ):
        ax.axvline(
            x - 0.5,
            linewidth=1.2,
        )

    ax.set_xlabel(
        "Nodes grouped by network → subnetwork"
    )
    ax.set_ylabel(ylabel)
    ax.set_title(title)

    fig.colorbar(
        im,
        ax=ax,
        shrink=0.8,
    )

    plt.show()


plot_pattern_matrix(
    make_true_motifs(0.0),
    "Separated endpoint",
    ylabel="True motif",
)

plot_pattern_matrix(
    make_true_motifs(1.0),
    "Overlapping endpoint",
    ylabel="True motif",
)



## 3. Multisubject simulation and normalization

Each subject has a perturbed version of the same group-level motifs. Every time point is generated from a sparse convex mixture of those motifs plus measurement noise.


In [ ]:

def simulate_multisubject_data(
    true_motifs,
    noise_sd=BASE_NOISE_SD,
    multiplicative_sd=BASE_MULTIPLICATIVE_SD,
    additive_sd=BASE_ADDITIVE_SD,
    seed=RNG_SEED,
):
    rng = np.random.default_rng(seed)

    K, V = true_motifs.shape

    raw_subjects = []
    true_weights = []
    subject_specific_motifs = []

    for s in range(N_SUBJECTS):
        multiplicative = (
            1.0
            + rng.normal(
                0,
                multiplicative_sd,
                size=(K, V),
            )
        )

        additive = rng.normal(
            0,
            additive_sd,
            size=(K, V),
        )

        subj_motifs = (
            true_motifs * multiplicative
            + additive
        )

        W = rng.dirichlet(
            np.full(K, DIRICHLET_ALPHA),
            size=T_PER_SUBJECT,
        )

        X = W @ subj_motifs

        X += rng.normal(
            0,
            noise_sd,
            size=X.shape,
        )

        raw_subjects.append(X)
        true_weights.append(W)
        subject_specific_motifs.append(subj_motifs)

    return {
        "raw_subjects": raw_subjects,
        "true_weights": true_weights,
        "subject_specific_motifs": subject_specific_motifs,
    }


def normalize_across_subjects(subject_list):
    subjects = [
        np.asarray(x, dtype=float)
        for x in subject_list
    ]

    stacked = np.vstack(subjects)

    mu = stacked.mean(
        axis=0,
        keepdims=True,
    )

    sd = (
        stacked.std(
            axis=0,
            keepdims=True,
        )
        + 1e-8
    )

    normalized = [
        ((x - mu) / sd).astype(
            np.float32,
            copy=False,
        )
        for x in subjects
    ]

    return normalized, mu, sd



## 4. Recovery, subspace, and mixing metrics

The benchmark deliberately uses several complementary metrics.

### One-to-one motif recovery
Optimal Hungarian assignment between true motifs and recovered dimensions.

### Subnetwork recovery
The same test after averaging each pattern within known subnetworks.

### PCA subspace recovery
Each true motif is projected into and reconstructed from the fitted PCA subspace. High subspace recovery with lower one-component recovery means PCA retained the information but rotated/mixed the generating motifs.

### Selectivity / purity
For each recovered dimension, absolute correlations with all true motifs are converted into a normalized squared-correlation profile.

- **Purity = 1 - normalized entropy**
- Higher purity means a dimension corresponds preferentially to one true motif.
- **Top1 minus Top2 selectivity** gives a simpler complementary measure.
- **Second/first mixing ratio** is lower when a dimension is less mixed.


In [ ]:

def pattern_corr_matrix(A, B, absolute=False):
    A = np.asarray(A, dtype=float)
    B = np.asarray(B, dtype=float)

    A0 = A - A.mean(
        axis=1,
        keepdims=True,
    )
    B0 = B - B.mean(
        axis=1,
        keepdims=True,
    )

    A0 /= (
        np.linalg.norm(
            A0,
            axis=1,
            keepdims=True,
        )
        + 1e-12
    )
    B0 /= (
        np.linalg.norm(
            B0,
            axis=1,
            keepdims=True,
        )
        + 1e-12
    )

    C = A0 @ B0.T

    if absolute:
        C = np.abs(C)

    return C


def optimal_match(true_patterns, recovered_patterns, absolute=False):
    C = pattern_corr_matrix(
        true_patterns,
        recovered_patterns,
        absolute=absolute,
    )

    true_idx, recovered_idx = linear_sum_assignment(-C)

    matched = C[
        true_idx,
        recovered_idx,
    ]

    return {
        "similarity": C,
        "true_idx": true_idx,
        "recovered_idx": recovered_idx,
        "matched": matched,
    }


def collapse_to_subnetworks(patterns):
    patterns = np.asarray(
        patterns,
        dtype=float,
    )

    n_subnetworks = (
        N_NETWORKS
        * SUBNETWORKS_PER_NETWORK
    )

    out = np.zeros(
        (
            patterns.shape[0],
            n_subnetworks,
        )
    )

    for subnet in range(n_subnetworks):
        lo = subnet * NODES_PER_SUBNETWORK
        hi = lo + NODES_PER_SUBNETWORK

        out[:, subnet] = (
            patterns[:, lo:hi]
            .mean(axis=1)
        )

    return out


def dimension_mixing_metrics(true_patterns, recovered_patterns):
    # recovered x true absolute pattern correlations
    C = pattern_corr_matrix(
        recovered_patterns,
        true_patterns,
        absolute=True,
    )

    squared = C ** 2
    probs = squared / (
        squared.sum(axis=1, keepdims=True)
        + 1e-12
    )

    entropy = -np.sum(
        probs * np.log(probs + 1e-12),
        axis=1,
    )

    entropy /= np.log(
        true_patterns.shape[0]
    )

    purity = 1.0 - entropy

    sorted_corr = np.sort(
        C,
        axis=1,
    )[:, ::-1]

    top1 = sorted_corr[:, 0]
    top2 = sorted_corr[:, 1]

    selectivity = top1 - top2

    mixing_ratio = top2 / (
        top1 + 1e-12
    )

    return {
        "mean_purity": np.mean(purity),
        "mean_selectivity": np.mean(selectivity),
        "mean_mixing_ratio": np.mean(mixing_ratio),
        "mean_best_abs_corr": np.mean(top1),
    }



## 5. Run one complete archetypal simulation

This function performs one matched MS-AA/PCA benchmark and returns a single summary row.


In [ ]:

def run_archetypal_simulation(
    experiment,
    level,
    rep,
    overlap,
    noise_sd,
    heterogeneity_scale,
    K_fit,
    seed,
    verbose=False,
):
    true_raw = make_true_motifs(overlap)

    multiplicative_sd = (
        BASE_MULTIPLICATIVE_SD
        * heterogeneity_scale
    )
    additive_sd = (
        BASE_ADDITIVE_SD
        * heterogeneity_scale
    )

    sim = simulate_multisubject_data(
        true_raw,
        noise_sd=noise_sd,
        multiplicative_sd=multiplicative_sd,
        additive_sd=additive_sd,
        seed=seed,
    )

    subjects, mu, sd = normalize_across_subjects(
        sim["raw_subjects"]
    )

    # Apply the same affine feature normalization to the true motifs.
    true_motifs = (
        true_raw - mu
    ) / sd

    X_stack = np.vstack(subjects)

    # ---------------- MS-AA ----------------
    msaa = fit_msaa(
        subjects,
        K=K_fit,
        analysis_type="spatial",
        seed=seed,
        verbose=verbose,
    )

    S_subjects = np.stack(
        [
            np.asarray(
                sub["S"],
                dtype=float,
            )
            for sub in msaa["results_subj"]
        ],
        axis=0,
    )

    S_group = S_subjects.mean(axis=0)

    Xhat_aa = np.vstack([
        reconstruct_from_archetypes(
            sub,
            "spatial",
            zscore=False,
        )
        for sub in msaa["results_subj"]
    ])

    aa_mse = np.mean(
        (X_stack - Xhat_aa) ** 2
    )

    # ---------------- PCA ----------------
    pca = PCA(
        n_components=K_fit,
        random_state=seed,
    )

    scores = pca.fit_transform(
        X_stack
    )

    Xhat_pca = pca.inverse_transform(
        scores
    )

    pca_mse = np.mean(
        (X_stack - Xhat_pca) ** 2
    )

    # ---------------- Ground-truth recovery ----------------
    aa_match = optimal_match(
        true_motifs,
        S_group,
        absolute=False,
    )

    pca_match = optimal_match(
        true_motifs,
        pca.components_,
        absolute=True,
    )

    true_sub = collapse_to_subnetworks(
        true_motifs
    )
    aa_sub = collapse_to_subnetworks(
        S_group
    )
    pca_sub = collapse_to_subnetworks(
        pca.components_
    )

    aa_sub_match = optimal_match(
        true_sub,
        aa_sub,
        absolute=False,
    )

    pca_sub_match = optimal_match(
        true_sub,
        pca_sub,
        absolute=True,
    )

    # ---------------- PCA subspace control ----------------
    true_pca_recon = pca.inverse_transform(
        pca.transform(
            true_motifs
        )
    )

    pca_subspace_r = np.array([
        np.corrcoef(
            true_motifs[k],
            true_pca_recon[k],
        )[0, 1]
        for k in range(true_motifs.shape[0])
    ])

    # ---------------- Mixing/purity ----------------
    aa_mix = dimension_mixing_metrics(
        true_motifs,
        S_group,
    )

    pca_mix = dimension_mixing_metrics(
        true_motifs,
        pca.components_,
    )

    return {
        "experiment": experiment,
        "level": float(level),
        "rep": int(rep),
        "seed": int(seed),
        "overlap": float(overlap),
        "true_mean_pairwise_abs_corr": mean_pairwise_abs_corr(
            true_motifs
        ),
        "noise_sd": float(noise_sd),
        "heterogeneity_scale": float(heterogeneity_scale),
        "K_fit": int(K_fit),
        "n_subjects": int(N_SUBJECTS),
        "t_per_subject": int(T_PER_SUBJECT),

        "aa_mse": float(aa_mse),
        "pca_mse": float(pca_mse),

        "aa_motif_recovery": float(
            np.mean(aa_match["matched"])
        ),
        "pca_motif_recovery": float(
            np.mean(pca_match["matched"])
        ),

        "aa_subnetwork_recovery": float(
            np.mean(aa_sub_match["matched"])
        ),
        "pca_subnetwork_recovery": float(
            np.mean(pca_sub_match["matched"])
        ),

        "pca_subspace_recovery": float(
            np.mean(pca_subspace_r)
        ),

        "aa_purity": float(
            aa_mix["mean_purity"]
        ),
        "pca_purity": float(
            pca_mix["mean_purity"]
        ),

        "aa_selectivity": float(
            aa_mix["mean_selectivity"]
        ),
        "pca_selectivity": float(
            pca_mix["mean_selectivity"]
        ),

        "aa_mixing_ratio": float(
            aa_mix["mean_mixing_ratio"]
        ),
        "pca_mixing_ratio": float(
            pca_mix["mean_mixing_ratio"]
        ),

        "aa_best_abs_corr": float(
            aa_mix["mean_best_abs_corr"]
        ),
        "pca_best_abs_corr": float(
            pca_mix["mean_best_abs_corr"]
        ),

        "aa_varexpl": float(
            msaa["varexpl"]
        ),
        "pca_explained_variance": float(
            np.sum(
                pca.explained_variance_ratio_
            )
        ),
        "msaa_elapsed_sec": float(
            msaa["elapsed"]
        ),
    }



## 6. Restart-safe checkpoint helpers

A simulation is considered complete by the tuple:

`(experiment, level, rep)`

Completed rows are never rerun unless you manually delete them from the checkpoint CSV.


In [ ]:

def load_results():
    if RESULTS_CSV.exists():
        df = pd.read_csv(RESULTS_CSV)
        print(
            f"Loaded {len(df)} completed simulations"
        )
        return df

    return pd.DataFrame()


def simulation_is_complete(df, experiment, level, rep):
    if df.empty:
        return False

    mask = (
        (df["experiment"] == experiment)
        & np.isclose(
            df["level"].astype(float),
            float(level),
        )
        & (df["rep"].astype(int) == int(rep))
    )

    return bool(mask.any())


def save_row(df, row):
    new = pd.DataFrame([row])

    if df.empty:
        out = new
    else:
        out = pd.concat(
            [df, new],
            ignore_index=True,
        )

    out = out.drop_duplicates(
        subset=[
            "experiment",
            "level",
            "rep",
        ],
        keep="last",
    )

    tmp = RESULTS_CSV.with_suffix(
        ".tmp.csv"
    )

    out.to_csv(
        tmp,
        index=False,
    )

    os.replace(
        tmp,
        RESULTS_CSV,
    )

    return out



# PART A — Primary experiment: motif overlap / nonorthogonality

This is the theoretical centerpiece.

Everything except overlap is held fixed. If our interpretation is correct, increasing motif overlap should primarily hurt the **one-component interpretability of PCA**, while the PCA subspace may continue to retain the underlying motif information.


In [ ]:

results_df = load_results()

for overlap in OVERLAP_LEVELS:
    for rep in range(N_REPS_MAIN):

        experiment = "overlap"
        level = overlap

        if simulation_is_complete(
            results_df,
            experiment,
            level,
            rep,
        ):
            print(
                f"SKIP overlap={overlap}, rep={rep}"
            )
            continue

        seed = int(
            np.random.SeedSequence(
                [RNG_SEED, 101, int(overlap * 1000), rep]
            ).generate_state(1)[0]
        )

        print(
            f"RUN overlap={overlap}, rep={rep}"
        )

        row = run_archetypal_simulation(
            experiment=experiment,
            level=level,
            rep=rep,
            overlap=overlap,
            noise_sd=BASE_NOISE_SD,
            heterogeneity_scale=1.0,
            K_fit=N_TRUE_MOTIFS,
            seed=seed,
            verbose=False,
        )

        results_df = save_row(
            results_df,
            row,
        )

print("Done with overlap experiment.")



## Primary overlap results


In [ ]:

def summarize_metric(df, x, metric):
    return (
        df
        .groupby(x)[metric]
        .agg(["mean", "sem"])
        .reset_index()
    )


overlap_df = results_df[
    results_df["experiment"] == "overlap"
].copy()

display(
    overlap_df.groupby("level")[
        [
            "aa_motif_recovery",
            "pca_motif_recovery",
            "pca_subspace_recovery",
            "aa_purity",
            "pca_purity",
            "aa_mse",
            "pca_mse",
        ]
    ].mean()
)


In [ ]:

# Figure 1: one-to-one motif recovery vs overlap
fig, ax = plt.subplots(figsize=(7, 4.5))

for metric, label in [
    ("aa_motif_recovery", "MS-AA archetype"),
    ("pca_motif_recovery", "PCA component"),
    ("pca_subspace_recovery", "PCA subspace"),
]:
    s = summarize_metric(
        overlap_df,
        "level",
        metric,
    )

    ax.errorbar(
        s["level"],
        s["mean"],
        yerr=s["sem"],
        marker="o",
        label=label,
    )

ax.set_xlabel("Motif-overlap parameter")
ax.set_ylabel("Ground-truth motif recovery")
ax.set_ylim(0, 1.05)
ax.set_title(
    "Representation vs disentanglement as true motifs overlap"
)
ax.legend()

plt.show()


In [ ]:

# Figure 2: recovered-dimension purity vs overlap
fig, ax = plt.subplots(figsize=(7, 4.5))

for metric, label in [
    ("aa_purity", "MS-AA"),
    ("pca_purity", "PCA"),
]:
    s = summarize_metric(
        overlap_df,
        "level",
        metric,
    )

    ax.errorbar(
        s["level"],
        s["mean"],
        yerr=s["sem"],
        marker="o",
        label=label,
    )

ax.set_xlabel("Motif-overlap parameter")
ax.set_ylabel("Recovered-dimension purity")
ax.set_ylim(0, 1.05)
ax.set_title(
    "Selectivity of recovered dimensions"
)
ax.legend()

plt.show()


In [ ]:

# Figure 3: reconstruction error vs overlap
fig, ax = plt.subplots(figsize=(7, 4.5))

for metric, label in [
    ("aa_mse", "MS-AA"),
    ("pca_mse", "PCA"),
]:
    s = summarize_metric(
        overlap_df,
        "level",
        metric,
    )

    ax.errorbar(
        s["level"],
        s["mean"],
        yerr=s["sem"],
        marker="o",
        label=label,
    )

ax.set_xlabel("Motif-overlap parameter")
ax.set_ylabel("Observed-data reconstruction MSE")
ax.set_title(
    "Reconstruction need not favor the more interpretable representation"
)
ax.legend()

plt.show()



# PART B — Noise robustness

Overlap is held at the strongly overlapping endpoint while measurement noise varies.


In [ ]:

results_df = load_results()

for noise_sd in NOISE_LEVELS:
    for rep in range(N_REPS_ROBUST):

        experiment = "noise"
        level = noise_sd

        if simulation_is_complete(
            results_df,
            experiment,
            level,
            rep,
        ):
            print(
                f"SKIP noise={noise_sd}, rep={rep}"
            )
            continue

        seed = int(
            np.random.SeedSequence(
                [RNG_SEED, 202, int(noise_sd * 1000), rep]
            ).generate_state(1)[0]
        )

        print(
            f"RUN noise={noise_sd}, rep={rep}"
        )

        row = run_archetypal_simulation(
            experiment=experiment,
            level=level,
            rep=rep,
            overlap=1.0,
            noise_sd=noise_sd,
            heterogeneity_scale=1.0,
            K_fit=N_TRUE_MOTIFS,
            seed=seed,
        )

        results_df = save_row(
            results_df,
            row,
        )

print("Done with noise experiment.")


In [ ]:

noise_df = results_df[
    results_df["experiment"] == "noise"
].copy()

fig, ax = plt.subplots(figsize=(7, 4.5))

for metric, label in [
    ("aa_motif_recovery", "MS-AA"),
    ("pca_motif_recovery", "PCA component"),
]:
    s = summarize_metric(
        noise_df,
        "level",
        metric,
    )

    ax.errorbar(
        s["level"],
        s["mean"],
        yerr=s["sem"],
        marker="o",
        label=label,
    )

ax.set_xlabel("Measurement noise SD")
ax.set_ylabel("One-to-one motif recovery")
ax.set_ylim(0, 1.05)
ax.set_title("Robustness to measurement noise")
ax.legend()

plt.show()



# PART C — Between-subject heterogeneity

The base subject perturbation is multiplied by the heterogeneity scale:

- 0 = no subject-specific motif perturbation,
- 1 = original simulation,
- 1.5 = stronger between-subject variability.


In [ ]:

results_df = load_results()

for scale in HETEROGENEITY_LEVELS:
    for rep in range(N_REPS_ROBUST):

        experiment = "heterogeneity"
        level = scale

        if simulation_is_complete(
            results_df,
            experiment,
            level,
            rep,
        ):
            print(
                f"SKIP heterogeneity={scale}, rep={rep}"
            )
            continue

        seed = int(
            np.random.SeedSequence(
                [RNG_SEED, 303, int(scale * 1000), rep]
            ).generate_state(1)[0]
        )

        print(
            f"RUN heterogeneity={scale}, rep={rep}"
        )

        row = run_archetypal_simulation(
            experiment=experiment,
            level=level,
            rep=rep,
            overlap=1.0,
            noise_sd=BASE_NOISE_SD,
            heterogeneity_scale=scale,
            K_fit=N_TRUE_MOTIFS,
            seed=seed,
        )

        results_df = save_row(
            results_df,
            row,
        )

print("Done with heterogeneity experiment.")


In [ ]:

hetero_df = results_df[
    results_df["experiment"] == "heterogeneity"
].copy()

fig, ax = plt.subplots(figsize=(7, 4.5))

for metric, label in [
    ("aa_motif_recovery", "MS-AA"),
    ("pca_motif_recovery", "PCA component"),
]:
    s = summarize_metric(
        hetero_df,
        "level",
        metric,
    )

    ax.errorbar(
        s["level"],
        s["mean"],
        yerr=s["sem"],
        marker="o",
        label=label,
    )

ax.set_xlabel("Subject-heterogeneity multiplier")
ax.set_ylabel("One-to-one motif recovery")
ax.set_ylim(0, 1.05)
ax.set_title(
    "Robustness to subject-specific spatial variability"
)
ax.legend()

plt.show()



# PART D — Sensitivity to fitted model order \(K\)

The true number of motifs remains 6 while the fitted model order varies.

For \(K<6\), perfect one-to-one recovery is impossible because there are fewer recovered dimensions than true motifs. The Hungarian score therefore reflects recovery of the best matchable subset.

For \(K>6\), extra dimensions may split or duplicate motifs. Purity and mixing metrics become especially useful here.


In [ ]:

results_df = load_results()

for K_fit in K_LEVELS:
    for rep in range(N_REPS_K):

        experiment = "K"
        level = K_fit

        if simulation_is_complete(
            results_df,
            experiment,
            level,
            rep,
        ):
            print(
                f"SKIP K={K_fit}, rep={rep}"
            )
            continue

        seed = int(
            np.random.SeedSequence(
                [RNG_SEED, 404, int(K_fit), rep]
            ).generate_state(1)[0]
        )

        print(
            f"RUN K={K_fit}, rep={rep}"
        )

        row = run_archetypal_simulation(
            experiment=experiment,
            level=level,
            rep=rep,
            overlap=1.0,
            noise_sd=BASE_NOISE_SD,
            heterogeneity_scale=1.0,
            K_fit=K_fit,
            seed=seed,
        )

        results_df = save_row(
            results_df,
            row,
        )

print("Done with K experiment.")


In [ ]:

k_df = results_df[
    results_df["experiment"] == "K"
].copy()

fig, ax = plt.subplots(figsize=(7, 4.5))

for metric, label in [
    ("aa_motif_recovery", "MS-AA"),
    ("pca_motif_recovery", "PCA component"),
]:
    s = summarize_metric(
        k_df,
        "level",
        metric,
    )

    ax.errorbar(
        s["level"],
        s["mean"],
        yerr=s["sem"],
        marker="o",
        label=label,
    )

ax.axvline(
    N_TRUE_MOTIFS,
    linestyle="--",
    alpha=0.6,
)

ax.set_xlabel("Fitted model order K")
ax.set_ylabel("One-to-one motif recovery")
ax.set_ylim(0, 1.05)
ax.set_title(
    "Sensitivity to model-order misspecification"
)
ax.legend()

plt.show()



# PART E — Orthogonal Gaussian negative control

Here the data are generated from ordinary orthogonal Gaussian latent factors rather than convex mixtures of extremal motifs.

This is a geometry explicitly favorable to PCA. A credible benchmark should **not** require MS-AA to win here.


In [ ]:

def simulate_gaussian_control(seed):
    rng = np.random.default_rng(seed)

    V = (
        N_NETWORKS
        * SUBNETWORKS_PER_NETWORK
        * NODES_PER_SUBNETWORK
    )

    Q, _ = np.linalg.qr(
        rng.normal(
            size=(V, N_TRUE_MOTIFS)
        )
    )

    true_factors = Q.T

    subjects = []

    for s in range(N_SUBJECTS):
        scores = rng.normal(
            size=(
                T_PER_SUBJECT,
                N_TRUE_MOTIFS,
            )
        )

        X = (
            scores @ true_factors
        )

        X += rng.normal(
            0,
            BASE_NOISE_SD,
            size=X.shape,
        )

        subjects.append(X)

    subjects_norm, mu, sd = (
        normalize_across_subjects(
            subjects
        )
    )

    # Feature scaling affects factor pattern amplitudes.
    true_transformed = (
        true_factors / sd
    )

    return (
        subjects_norm,
        true_transformed,
    )


def run_gaussian_control(rep, seed):
    subjects, true_factors = (
        simulate_gaussian_control(seed)
    )

    X_stack = np.vstack(subjects)

    msaa = fit_msaa(
        subjects,
        K=N_TRUE_MOTIFS,
        analysis_type="spatial",
        seed=seed,
    )

    S_group = np.mean(
        np.stack(
            [
                np.asarray(
                    sub["S"],
                    dtype=float,
                )
                for sub in msaa[
                    "results_subj"
                ]
            ],
            axis=0,
        ),
        axis=0,
    )

    Xhat_aa = np.vstack([
        reconstruct_from_archetypes(
            sub,
            "spatial",
            zscore=False,
        )
        for sub in msaa["results_subj"]
    ])

    pca = PCA(
        n_components=N_TRUE_MOTIFS,
        random_state=seed,
    ).fit(X_stack)

    Xhat_pca = pca.inverse_transform(
        pca.transform(X_stack)
    )

    aa_match = optimal_match(
        true_factors,
        S_group,
        absolute=True,
    )

    pca_match = optimal_match(
        true_factors,
        pca.components_,
        absolute=True,
    )

    return {
        "experiment": "gaussian",
        "level": 0.0,
        "rep": int(rep),
        "seed": int(seed),
        "overlap": np.nan,
        "true_mean_pairwise_abs_corr": mean_pairwise_abs_corr(
            true_factors
        ),
        "noise_sd": BASE_NOISE_SD,
        "heterogeneity_scale": np.nan,
        "K_fit": N_TRUE_MOTIFS,
        "n_subjects": N_SUBJECTS,
        "t_per_subject": T_PER_SUBJECT,

        "aa_mse": np.mean(
            (X_stack - Xhat_aa) ** 2
        ),
        "pca_mse": np.mean(
            (X_stack - Xhat_pca) ** 2
        ),

        "aa_motif_recovery": np.mean(
            aa_match["matched"]
        ),
        "pca_motif_recovery": np.mean(
            pca_match["matched"]
        ),

        "aa_subnetwork_recovery": np.nan,
        "pca_subnetwork_recovery": np.nan,
        "pca_subspace_recovery": np.nan,

        "aa_purity": np.nan,
        "pca_purity": np.nan,
        "aa_selectivity": np.nan,
        "pca_selectivity": np.nan,
        "aa_mixing_ratio": np.nan,
        "pca_mixing_ratio": np.nan,
        "aa_best_abs_corr": np.nan,
        "pca_best_abs_corr": np.nan,

        "aa_varexpl": msaa["varexpl"],
        "pca_explained_variance": np.sum(
            pca.explained_variance_ratio_
        ),
        "msaa_elapsed_sec": msaa["elapsed"],
    }


In [ ]:

results_df = load_results()

for rep in range(N_REPS_GAUSSIAN):

    experiment = "gaussian"
    level = 0.0

    if simulation_is_complete(
        results_df,
        experiment,
        level,
        rep,
    ):
        print(
            f"SKIP gaussian rep={rep}"
        )
        continue

    seed = int(
        np.random.SeedSequence(
            [RNG_SEED, 505, rep]
        ).generate_state(1)[0]
    )

    print(
        f"RUN gaussian rep={rep}"
    )

    row = run_gaussian_control(
        rep=rep,
        seed=seed,
    )

    results_df = save_row(
        results_df,
        row,
    )

print("Done with Gaussian control.")


In [ ]:

gaussian_df = results_df[
    results_df["experiment"] == "gaussian"
].copy()

display(
    gaussian_df[
        [
            "aa_motif_recovery",
            "pca_motif_recovery",
            "aa_mse",
            "pca_mse",
        ]
    ].agg(["mean", "sem"])
)



# PART F — Representative high-overlap visualization

The repeated statistics should be primary. This section reruns one representative strongly overlapping simulation so we can directly visualize:

1. the true motifs,
2. optimally matched MS-AA archetypes,
3. optimally matched PCA components,
4. the corresponding similarity matrices.

This is useful for a figure panel but should not substitute for the repeated benchmark.


In [ ]:

REPRESENTATIVE_SEED = 20260907

true_raw = make_true_motifs(1.0)

sim = simulate_multisubject_data(
    true_raw,
    noise_sd=BASE_NOISE_SD,
    multiplicative_sd=BASE_MULTIPLICATIVE_SD,
    additive_sd=BASE_ADDITIVE_SD,
    seed=REPRESENTATIVE_SEED,
)

subjects, mu, sd = normalize_across_subjects(
    sim["raw_subjects"]
)

true_motifs = (
    true_raw - mu
) / sd

msaa = fit_msaa(
    subjects,
    K=N_TRUE_MOTIFS,
    analysis_type="spatial",
    seed=REPRESENTATIVE_SEED,
    verbose=True,
)

S_group = np.mean(
    np.stack(
        [
            np.asarray(
                sub["S"],
                dtype=float,
            )
            for sub in msaa[
                "results_subj"
            ]
        ],
        axis=0,
    ),
    axis=0,
)

X_stack = np.vstack(subjects)

pca = PCA(
    n_components=N_TRUE_MOTIFS,
    random_state=REPRESENTATIVE_SEED,
).fit(X_stack)

aa_match = optimal_match(
    true_motifs,
    S_group,
    absolute=False,
)

pca_match = optimal_match(
    true_motifs,
    pca.components_,
    absolute=True,
)

AA_ORDERED = S_group[
    aa_match["recovered_idx"]
].copy()

PCA_ORDERED = pca.components_[
    pca_match["recovered_idx"]
].copy()

for i in range(len(PCA_ORDERED)):
    r = np.corrcoef(
        true_motifs[i],
        PCA_ORDERED[i],
    )[0, 1]

    if r < 0:
        PCA_ORDERED[i] *= -1


plot_pattern_matrix(
    true_motifs,
    "Ground truth: overlapping distributed motifs",
    ylabel="True motif",
)

plot_pattern_matrix(
    AA_ORDERED,
    "MS-AA archetypes matched to ground truth",
    ylabel="Matched archetype",
)

plot_pattern_matrix(
    PCA_ORDERED,
    "PCA components matched to ground truth",
    ylabel="Matched component",
)


In [ ]:

def plot_similarity_matrix(C, title):
    fig, ax = plt.subplots(
        figsize=(5.5, 4.5)
    )

    im = ax.imshow(
        C,
        vmin=0,
        vmax=1,
        aspect="auto",
    )

    ax.set_xlabel("Recovered dimension")
    ax.set_ylabel("True motif")
    ax.set_title(title)

    fig.colorbar(
        im,
        ax=ax,
        label="Absolute pattern similarity",
    )

    plt.show()


plot_similarity_matrix(
    np.abs(
        pattern_corr_matrix(
            true_motifs,
            S_group,
        )
    ),
    "True motifs ↔ MS-AA archetypes",
)

plot_similarity_matrix(
    pattern_corr_matrix(
        true_motifs,
        pca.components_,
        absolute=True,
    ),
    "True motifs ↔ PCA components",
)



# PART G — Final summary table and candidate manuscript quantities


In [ ]:

results_df = load_results()

summary_rows = []

for experiment in [
    "overlap",
    "noise",
    "heterogeneity",
    "K",
    "gaussian",
]:
    sub = results_df[
        results_df["experiment"] == experiment
    ]

    if len(sub) == 0:
        continue

    summary_rows.append({
        "experiment": experiment,
        "n_simulations": len(sub),
        "mean_AA_motif_recovery": sub[
            "aa_motif_recovery"
        ].mean(),
        "mean_PCA_component_recovery": sub[
            "pca_motif_recovery"
        ].mean(),
        "mean_AA_minus_PCA_recovery": (
            sub["aa_motif_recovery"]
            - sub["pca_motif_recovery"]
        ).mean(),
        "mean_AA_MSE": sub[
            "aa_mse"
        ].mean(),
        "mean_PCA_MSE": sub[
            "pca_mse"
        ].mean(),
    })

summary_df = pd.DataFrame(
    summary_rows
)

display(summary_df)



# How to interpret the completed benchmark

The strongest manuscript result would be a **geometry-specific dissociation**:

### Archetypal/extremal regime
As true motifs become more overlapping/nonorthogonal:

- PCA reconstruction remains strong;
- PCA-subspace recovery of the true motifs remains strong;
- individual PCA components become less one-to-one with the generating motifs and/or less pure;
- MS-AA archetypes retain stronger motif and subnetwork correspondence.

### Robustness
The MS-AA interpretability advantage persists across reasonable measurement noise and subject heterogeneity.

### Model order
The result is not limited to an implausibly exact choice of fitted \(K\).

### Gaussian control
When the actual generative geometry is orthogonal Gaussian latent variation, PCA matches or exceeds MS-AA.

If those patterns hold across repeated simulations, the claim should be framed narrowly:

> **MS-AA is not generically superior to PCA as a low-dimensional approximation. Rather, when data are organized as mixtures of nonorthogonal extremal network motifs, MS-AA can recover the identity and network composition of those motifs more directly, even when PCA preserves the same information within its low-dimensional subspace.**

That is the simulation result most directly aligned with the empirical manuscript's distinction between **generic compressibility** and **interpretable extremal motif structure**.
